## Vector Store and Retriever

In this notebook we are going to learn how to store the documents in a vector database and then retrieve them using a retriever. We will be using the `Chroma` vector store for this purpose. We'll also learn how to chain this retriever with a language model to create a question-answering system.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

## Documents

Langchain implements a Document class that is used to represent a piece of text. It has two attributes: `page_content` and `metadata`. The `page_content` attribute is the text of the document, while the `metadata` attribute is a dictionary that can contain any additional information about the document.

In [2]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="TVK promises a monthly financial assistance of ₹2,500 to women heads of households under the age of 60 to ensure economic independence.",
        metadata={"source": "tvk-manifesto-2026", "category": "Women's Welfare"},
    ),
    Document(
        page_content="To support households with rising costs, the party guarantees six free LPG cylinders per year for every family.",
        metadata={"source": "tvk-manifesto-2026", "category": "Women's Welfare"},
    ),
    Document(
        page_content="The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.",
        metadata={"source": "tvk-manifesto-2026", "category": "Women's Welfare"},
    ),
    Document(
        page_content="The party pledges to abolish the National Eligibility-cum-Entrance Test (NEET) and move education from the Concurrent List back to the State List to restore state rights.",
        metadata={"source": "tvk-manifesto-2026", "category": "Education"},
    ),
    Document(
        page_content="TVK aims to establish 100 'Kamarajar Model Residential Schools' in each district to provide high-quality education to students from poor backgrounds.",
        metadata={"source": "tvk-manifesto-2026", "category": "Education"},
    ),
    Document(
        page_content="For the youth, the party promises to create 5 lakh government jobs and provide an unemployment allowance of ₹4,000 for graduates and ₹2,500 for diploma holders.",
        metadata={"source": "tvk-manifesto-2026", "category": "Employment"},
    ),
    Document(
        page_content="Agricultural reform plans include a complete waiver of crop loans for small farmers owning less than 5 acres and a 50% waiver for those with larger holdings.",
        metadata={"source": "tvk-manifesto-2026", "category": "Agriculture"},
    ),
    Document(
        page_content="The party ensures Minimum Support Prices (MSP) of ₹3,500 per quintal for paddy and ₹4,500 per ton for sugarcane to support farmer livelihoods.",
        metadata={"source": "tvk-manifesto-2026", "category": "Agriculture"},
    ),
    Document(
        page_content="TVK commits to providing health insurance coverage of up to ₹25 lakh per family to ensure accessible healthcare for all citizens.",
        metadata={"source": "tvk-manifesto-2026", "category": "Healthcare"},
    ),
    Document(
        page_content="A key infrastructure promise is the provision of 200 units of free electricity bi-monthly for domestic users to reduce household utility burdens.",
        metadata={"source": "tvk-manifesto-2026", "category": "Public Services"},
    ),
    Document(
        page_content="The party's ideology is rooted in 'Secular Social Justice', upholding equality for all as articulated in the slogan 'Pirappokkum Ella Uyirkkum' (All are equal by birth).",
        metadata={"source": "tvk-manifesto-2026", "category": "Ideology"},
    ),
    Document(
        page_content="TVK supports a two-language policy (Tamil and English) and opposes the imposition of Hindi, aiming to protect the Tamil language and culture.",
        metadata={"source": "tvk-manifesto-2026", "category": "Language Policy"},
    ),
    Document(
        page_content="Administrative reforms include establishing a branch of the State Secretariat in Madurai to decentralize power and improve access for southern districts.",
        metadata={"source": "tvk-manifesto-2026", "category": "Governance"},
    ),
    Document(
        page_content="The party takes a strict stance against drugs, promising a 'Drug-Free Tamil Nadu' through the formation of special anti-drug squads in every district.",
        metadata={"source": "tvk-manifesto-2026", "category": "Law & Order"},
    ),
    Document(
        page_content="To empower local industries, the manifesto sets a goal of building a $1.5 trillion state economy by 2036 through industrial modernization and MSME support.",
        metadata={"source": "tvk-manifesto-2026", "category": "Economy"},
    ),
]

In [3]:
documents

[Document(metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='TVK promises a monthly financial assistance of ₹2,500 to women heads of households under the age of 60 to ensure economic independence.'),
 Document(metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='To support households with rising costs, the party guarantees six free LPG cylinders per year for every family.'),
 Document(metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.'),
 Document(metadata={'source': 'tvk-manifesto-2026', 'category': 'Education'}, page_content='The party pledges to abolish the National Eligibility-cum-Entrance Test (NEET) and move education from the Concurrent List back to the State List to restore state rights.'),
 Document(metadata={'source

## Vector Store and Retriever

To store the documents into a vector database we need to use an embedding model to convert the document into vectors. We are going to use Hugging Face's `sentence-transformers/all-MiniLM-L6-v2` model for this purpose. We will then use the `Chroma` vector store to store the vectors and create a retriever to retrieve the relevant documents based on a query.

In [5]:
## VectorStores

import groq
from openai import embeddings
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key = groq_api_key, model = "openai/gpt-oss-120b")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

from langchain_chroma import Chroma
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings
)

vector_store

e:\TCS\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1337.32it/s]


## Search

Let's cover basic vector similarity search

In [12]:
vector_store.similarity_search("What are the promises for farmers?", k=3)

[Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'source': 'tvk-manifesto-2026', 'category': 'Agriculture'}, page_content='Agricultural reform plans include a complete waiver of crop loans for small farmers owning less than 5 acres and a 50% waiver for those with larger holdings.'),
 Document(id='48639d1c-522e-4663-8b18-88f4b41cc581', metadata={'category': 'Agriculture', 'source': 'tvk-manifesto-2026'}, page_content='The party ensures Minimum Support Prices (MSP) of ₹3,500 per quintal for paddy and ₹4,500 per ton for sugarcane to support farmer livelihoods.'),
 Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'category': "Women's Welfare", 'source': 'tvk-manifesto-2026'}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.')]

In [11]:
## Async Query - The Execution does not blocks the main thread and allows other operations to run concurrently while waiting for the result.

await vector_store.asimilarity_search("What are the promises for farmers?", k=3)

[Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'source': 'tvk-manifesto-2026', 'category': 'Agriculture'}, page_content='Agricultural reform plans include a complete waiver of crop loans for small farmers owning less than 5 acres and a 50% waiver for those with larger holdings.'),
 Document(id='48639d1c-522e-4663-8b18-88f4b41cc581', metadata={'source': 'tvk-manifesto-2026', 'category': 'Agriculture'}, page_content='The party ensures Minimum Support Prices (MSP) of ₹3,500 per quintal for paddy and ₹4,500 per ton for sugarcane to support farmer livelihoods.'),
 Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.')]

In [13]:
vector_store.similarity_search_with_score("What are the promises for farmers?", k=3)

[(Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'category': 'Agriculture', 'source': 'tvk-manifesto-2026'}, page_content='Agricultural reform plans include a complete waiver of crop loans for small farmers owning less than 5 acres and a 50% waiver for those with larger holdings.'),
  0.9871728420257568),
 (Document(id='48639d1c-522e-4663-8b18-88f4b41cc581', metadata={'category': 'Agriculture', 'source': 'tvk-manifesto-2026'}, page_content='The party ensures Minimum Support Prices (MSP) of ₹3,500 per quintal for paddy and ₹4,500 per ton for sugarcane to support farmer livelihoods.'),
  1.3087116479873657),
 (Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.'),
  1.386324405670166)]

In [14]:
await vector_store.asimilarity_search_with_score("What are the promises for farmers?", k=3)

[(Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'source': 'tvk-manifesto-2026', 'category': 'Agriculture'}, page_content='Agricultural reform plans include a complete waiver of crop loans for small farmers owning less than 5 acres and a 50% waiver for those with larger holdings.'),
  0.9871728420257568),
 (Document(id='48639d1c-522e-4663-8b18-88f4b41cc581', metadata={'source': 'tvk-manifesto-2026', 'category': 'Agriculture'}, page_content='The party ensures Minimum Support Prices (MSP) of ₹3,500 per quintal for paddy and ₹4,500 per ton for sugarcane to support farmer livelihoods.'),
  1.3087116479873657),
 (Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'category': "Women's Welfare", 'source': 'tvk-manifesto-2026'}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.'),
  1.386324405670166)]

## Retrievers

Langchain VectorStore objects dont subclass Runnable. Which means they dont have the `invoke` method. So it can't be integrated in to LCEL chains

LangChain Retrivers are Runnables so they can be integrated in to LCEL chains. They are a wrapper around VectorStores that provide a more convenient interface for searching and retrieving documents. They also allow you to specify additional parameters such as the number of results to return and the minimum similarity score.

In [23]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vector_store.similarity_search).bind(k=3)
retriever.batch(['women', 'farmers', 'education'])

[[Document(id='82bde055-9f20-41e1-ac9d-879c2873f87e', metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='TVK promises a monthly financial assistance of ₹2,500 to women heads of households under the age of 60 to ensure economic independence.'),
  Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'source': 'tvk-manifesto-2026', 'category': "Women's Welfare"}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.'),
  Document(id='6709a00e-b432-4fb9-b68f-04b2e8263c57', metadata={'source': 'tvk-manifesto-2026', 'category': 'Ideology'}, page_content="The party's ideology is rooted in 'Secular Social Justice', upholding equality for all as articulated in the slogan 'Pirappokkum Ella Uyirkkum' (All are equal by birth).")],
 [Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'source': 'tvk-manifesto-2026', 'category': '

Instead of RunnableLambda we can also use vectorstore.as_retriever() to create a retriever that can be used with other components in Langchain, such as a RetrievalQA chain.

In [20]:
vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

retriever.batch(['women', 'farmers', 'education'])

[[Document(id='82bde055-9f20-41e1-ac9d-879c2873f87e', metadata={'category': "Women's Welfare", 'source': 'tvk-manifesto-2026'}, page_content='TVK promises a monthly financial assistance of ₹2,500 to women heads of households under the age of 60 to ensure economic independence.'),
  Document(id='25f807f8-0306-4ba8-9595-0fc74a30b199', metadata={'category': "Women's Welfare", 'source': 'tvk-manifesto-2026'}, page_content='The manifesto includes a marriage assistance scheme providing one sovereign (8 grams) of gold and a silk saree to brides from economically weaker sections.'),
  Document(id='6709a00e-b432-4fb9-b68f-04b2e8263c57', metadata={'category': 'Ideology', 'source': 'tvk-manifesto-2026'}, page_content="The party's ideology is rooted in 'Secular Social Justice', upholding equality for all as articulated in the slogan 'Pirappokkum Ella Uyirkkum' (All are equal by birth).")],
 [Document(id='cdbb95b6-383a-46b0-8602-a784e1d3e5d9', metadata={'source': 'tvk-manifesto-2026', 'category': '

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer the question with provided context only.

{question}

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant who helps understand the users understand our political party goal."),
        ("human", message),
    ]
)

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | prompt | llm

response = rag_chain.invoke("What are the benefits for the Government")
print(response.content)

**Benefits for the Government (as implied by the manifesto promises)**  

1. **Employment creation** – By promising 5 lakh government jobs for youth, the government can showcase a tangible record of job generation, which strengthens its reputation for addressing unemployment.  

2. **Targeted unemployment allowance** – Providing ₹4,000 to graduates and ₹2,500 to diploma holders positions the government as a direct supporter of young job‑seekers, potentially increasing public goodwill and political support among educated youth.  

3. **Economic expansion** – The goal of building a **$1.5 trillion state economy by 2036** through industrial modernization and MSME support gives the government a clear, high‑profile growth target. Achieving this would expand the tax base, increase fiscal resources, and demonstrate effective economic stewardship.  

4. **Industrial modernization & MSME boost** – Supporting local industries and micro‑small‑medium enterprises (MSMEs) can lead to higher producti